# LeetCode 94 — Binary Tree Inorder Traversal
Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Binary Tree · DFS · Recursion · Stack  
**Bài gốc:** https://leetcode.com/problems/binary-tree-inorder-traversal/

## Bạn sẽ học được gì?
1. Tự tạo Binary Tree qua TreeNode, root, left, right.
2. Phân biệt thứ tự Left → Node → Right với các kiểu duyệt khác.
3. Thấy cách giải đệ quy trực giác từ cây con.
4. Thực hành stack để mô phỏng lời gọi đệ quy.
5. Trace từng push/pop và nộp inorderTraversal.

**Cách học:** đọc Markdown → đoán kết quả của cell code tiếp theo → chạy cell → so sánh. Phần TODO chỉ để luyện tập, không tự chạy bài tập chưa hoàn thiện.

## 1. Đề bài — duyệt cây theo thứ tự giữa

Cho **root** của một **binary tree (cây nhị phân)**: mỗi node có tối đa **hai con** `left` và `right`. Trả về list các giá trị theo thứ tự **inorder**:

**Duyệt cây con TRÁI → thăm NODE HIỆN TẠI → duyệt cây con PHẢI.**

Ví dụ (mũi tên biểu diễn quan hệ cha-con, không phải danh sách):

```text
    1
     \
      2
     /
    3
```

- Cây con trái của 1 rỗng → ghi 1.
- Vào cây con phải là 2; cây con trái của 2 là 3 → ghi 3 trước khi ghi 2.
- Kết quả `[1,3,2]`.

**Cảnh báo:** inorder KHÔNG có nghĩa là cứ ghi giá trị theo thứ tự tăng dần. Chỉ với **BST hợp lệ** mới thường được kết quả tăng dần.

## 2. `TreeNode`: mỗi node có ba ô

- `val`: giá trị trong node.
- `left`: tham chiếu đến node con trái, có thể là `None`.
- `right`: tham chiếu đến node con phải, có thể là `None`.

`root` là node gốc. `None` nghĩa là nhánh không tồn tại. LeetCode sẽ cung cấp `TreeNode`; notebook tự tạo để học trên Python 3.

In [ ]:
from __future__ import annotations
from typing import Optional

class TreeNode:
    def __init__(self, val: int = 0, left: Optional["TreeNode"] = None,
                 right: Optional["TreeNode"] = None):
        self.val = val
        self.left = left
        self.right = right

root = TreeNode(1)
root.right = TreeNode(2)
root.right.left = TreeNode(3)
print("root:",root.val)
print("root.right:",root.right.val)
print("root.right.left:",root.right.left.val)
print("root.left is None?",root.left is None)

## 3. Python nền tảng: tham chiếu, None và điều kiện

`node.left` không phải là danh sách; nó **trỏ đến một node**. Viết `node.left = TreeNode(7)` làm xuất hiện một nhánh trái mới.

Khi đọc `node.left.val`, phải chắc chắn `node.left` không phải `None`. Một node lá là node có **cả left và right đều None**.

In [ ]:
leaf=TreeNode(5)
print("Lá?",leaf.left is None and leaf.right is None)
leaf.left=TreeNode(1)
print("Sau khi thêm con trái, còn là lá?",leaf.left is None and leaf.right is None)
print("Giá trị con trái:",leaf.left.val)

## 4. Nhắc lại: Left → NODE → Right, không phải Node trước

Lấy cây đầy đủ:

```text
      4
     / \
    2   6
   / \ / \
  1  3 5  7
```

Ở gốc 4, **không ghi 4 trước**; phải duyệt toàn bộ cây trái `[1,2,3]`, rồi ghi 4, cuối cùng ghi `[5,6,7]`. Kết quả `[1,2,3,4,5,6,7]`.

**Preorder:** node → trái → phải. **Postorder:** trái → phải → node. Chỉ bài này yêu cầu **inorder**.

In [ ]:
balanced = TreeNode(4,
    TreeNode(2,TreeNode(1),TreeNode(3)),
    TreeNode(6,TreeNode(5),TreeNode(7)))
print("Node gốc:",balanced.val)
print("Con trái:",balanced.left.val,"; con phải:",balanced.right.val)

## 5. Cách tự nhiên: đệ quy (recursion) theo định nghĩa

Định nghĩa inorder **tự chứa một bài toán nhỏ hơn**: inorder của cây con trái và inorder của cây con phải. Vì vậy, đệ quy là cách biểu diễn trực giác nhất.

`visit(node)`:
1. Nếu `node is None`, **không làm gì** rồi trở về.
2. `visit(node.left)`.
3. `result.append(node.val)`.
4. `visit(node.right)`.

**Append** thêm giá trị vào cuối list kết quả. Ta tạo `result=[]` một lần để các lời gọi cùng đóng góp.

In [ ]:
def inorder_recursive(root: Optional[TreeNode]) -> list[int]:
    result = []
    def visit(node):
        if node is None:
            return
        visit(node.left)        # TRÁI
        result.append(node.val) # NODE
        visit(node.right)       # PHẢI
    visit(root)
    return result

print("Ví dụ [1,null,2,3]:",inorder_recursive(root))
print("Cây cân bằng:",inorder_recursive(balanced))

## 6. Python nền tảng: gọi hàm, `return`, biến trong hàm lồng

`visit` là một hàm **được định nghĩa bên trong** `inorder_recursive`; nó có thể dùng biến `result` của hàm bao ngoài để `append`.

Khi `visit(None)` chạy `return` → **quay về** lời gọi trước, không thêm giá trị. Mỗi lời gọi đang dang dở được ghi nhớ trên **call stack (ngăn xếp lời gọi)**.

In [ ]:
def count_nodes_recursive(node):
    if node is None:
        return 0
    return 1 + count_nodes_recursive(node.left) + count_nodes_recursive(node.right)

print("Số node cây ví dụ:",count_nodes_recursive(root))
print("Số node cây cân bằng:",count_nodes_recursive(balanced))

## 7. Trace lời gọi đệ quy — nhìn đúng thời điểm ghi node

Với cây `1 → phải 2 → trái 3`: `visit(1)` gọi trái (None) → **ghi 1** → gọi phải 2 → sang trái 3 → **ghi 3** → trở về 2 → **ghi 2**.

Ta in rõ **bước vào / bước ghi / bước ra**. Số khoảng trắng biểu diễn độ sâu đệ quy.

In [ ]:
def trace_inorder_recursive(root):
    out=[]
    def visit(node,depth=0):
        if node is None:
            print("  "*depth + "gặp None -> quay về")
            return
        print("  "*depth + f"vào node {node.val}")
        visit(node.left,depth+1)
        out.append(node.val)
        print("  "*depth + f"GHI {node.val} -> {out}")
        visit(node.right,depth+1)
    visit(root)
    return out

trace_inorder_recursive(root)

## 8. Vì sao cần học thêm stack tường minh?

Đệ quy đã đúng, Time O(n), nhưng dùng **call stack** của Python. Nếu cây rất sâu, đệ quy có thể vướng giới hạn số lời gọi.

Ta có thể mô phỏng bằng **stack tự tạo** trong Python: list với hai thao tác:
- `stack.append(node)` **đẩy** node lên đỉnh (*push*).
- `stack.pop()` **lấy/bỏ** node ở đỉnh (*pop*).

Stack tuân theo **LIFO**: node vào sau sẽ ra trước. Đây không phải cách cải thiện Time Big-O, mà là **cách kiểm soát trạng thái duyệt** không cần đệ quy.

In [ ]:
stack=[]
stack.append("gốc")
stack.append("con trái")
print("đỉnh stack đang là:",stack[-1])
print("pop:",stack.pop())
print("pop:",stack.pop())
print("stack còn:",stack)

## 9. Từ đệ quy sang stack: quy tắc thăm một node

Để duyệt **trái trước**, ta làm:
1. Nếu `current` còn node, **đẩy** nó vào stack rồi `current=current.left`.
2. Nếu `current=None` nhưng stack còn, **pop** node gần nhất → đây là node có cây trái đã đi xong, nên **ghi giá trị**.
3. Sau khi ghi, chuyển `current` sang **cây con phải** của node đó.

Vòng lặp chỉ dừng khi **không còn current VÀ stack rỗng**. Điều kiện `or` rất quan trọng: `while current is not None or stack:`.

In [ ]:
stack=[]
current=root
while current is not None:
    stack.append(current)
    print("push",current.val)
    current=current.left
print("Sau khi đi trái hết, stack values:",[node.val for node in stack])
current=stack.pop()
print("pop",current.val,"=> ghi giá trị rồi đi cây phải",current.right.val if current.right else None)

## 10. Thuật toán chính — inorder bằng stack

`while current is not None or stack:` bao gồm cả tình huống current rỗng nhưng vẫn còn node chờ ghi trong stack.

Hai nhánh khác nhau: `if current` là **đi trái và đẩy**, `else` là **pop, ghi, đi phải**.

In [ ]:
def inorder_iterative(root: Optional[TreeNode]) -> list[int]:
    result=[]
    stack=[]
    current=root
    while current is not None or stack:
        if current is not None:
            stack.append(current)
            current=current.left
        else:
            current=stack.pop()
            result.append(current.val)
            current=current.right
    return result

print("Ví dụ cây 1-2-3:",inorder_iterative(root))
print("Cây cân bằng:",inorder_iterative(balanced))

## 11. Bảng trace từ ví dụ `1 → phải 2 → trái 3`

| Lượt | current trước | stack trước (dưới → đỉnh) | Quyết định | result sau |
|---:|---|---|---|---|
| 1 | 1 | [] | push 1; sang trái=None | [] |
| 2 | None | [1] | pop 1; ghi 1; sang phải=2 | [1] |
| 3 | 2 | [] | push 2; sang trái=3 | [1] |
| 4 | 3 | [2] | push 3; sang trái=None | [1] |
| 5 | None | [2,3] | pop 3; ghi 3 | [1,3] |
| 6 | None | [2] | pop 2; ghi 2 | [1,3,2] |

Stack rỗng, current=None: dừng. **Không pop một node trước khi hoàn thành nhánh trái của nó.**

In [ ]:
def trace_inorder_iterative(root):
    stack=[]; current=root; result=[]; step=0
    while current is not None or stack:
        step+=1
        before_current=current.val if current is not None else None
        before_stack=[node.val for node in stack]
        if current is not None:
            action=f"push {current.val}; đi trái"
            stack.append(current)
            current=current.left
        else:
            current=stack.pop()
            action=f"pop {current.val}; GHI; đi phải"
            result.append(current.val)
            current=current.right
        print(f"{step:2}: current={before_current}, stack={before_stack} -> {action}; result={result}")
    print("TRẢ VỀ",result)
    return result

trace_inorder_iterative(root)

## 12. Vì sao cả hai thuật toán đều đúng?

**Đệ quy:** trước khi `append(node.val)`, ta đã duyệt hết cây trái; sau append mới duyệt cây phải. Đúng ngay theo định nghĩa inorder.

**Stack:** các node đang ở trong stack là những node **đã đi vào nhưng chưa được ghi**, đang chờ hoàn thành cây trái. Khi pop một node, cây trái của nó đã được xử lý; ta ghi node rồi chuyển sang cây phải. Do đó thứ tự luôn **trái → node → phải**, không bỏ sót node nào.

## 13. LeetCode: class Solution chuẩn

Nộp phương thức `inorderTraversal(self, root: Optional[TreeNode]) -> list[int]`. LeetCode tự cấp TreeNode; notebook đã khai báo sẵn để chạy local. Không cần in trace khi submit.

In [ ]:
class Solution:
    def inorderTraversal(self, root: Optional[TreeNode]) -> list[int]:
        result=[]
        stack=[]
        current=root
        while current is not None or stack:
            if current is not None:
                stack.append(current)
                current=current.left
            else:
                current=stack.pop()
                result.append(current.val)
                current=current.right
        return result

## 14. Test rỗng, một node, cây lệch trái/phải, cây cân bằng

Ta tạo cây bằng cách gọi `TreeNode` rõ ràng để **không phụ thuộc vào một trình chuyển mảng level-order chưa giải thích**. Ví dụ `[1,null,2,3]` được xây dưới dạng TreeNode ở đầu notebook.

In [ ]:
left_chain=TreeNode(3,TreeNode(2,TreeNode(1)))
right_chain=TreeNode(1,None,TreeNode(2,None,TreeNode(3)))
tests_94=[
    (None, []),
    (TreeNode(5),[5]),
    (root,[1,3,2]),
    (balanced,[1,2,3,4,5,6,7]),
    (left_chain,[1,2,3]),
    (right_chain,[1,2,3]),
    (TreeNode(2,TreeNode(-1),TreeNode(9)),[-1,2,9]),
]
for node,expected in tests_94:
    assert Solution().inorderTraversal(node)==expected,expected
    assert inorder_recursive(node)==expected
print("PASS",len(tests_94),"cây mẫu / biên")

## 15. Đối chiếu nhiều hình dạng cây một cách độc lập

Với mỗi `n`, tự xây một cây lệch trái, lệch phải và cây nhị phân gần cân bằng. Dù **hình dạng** khác nhau, cả đệ quy và stack phải cho cùng kết quả khi duyệt đúng các node.

In [ ]:
def make_left_chain(values):
    head=None
    for v in values:
        head=TreeNode(v,head)
    return head

def make_right_chain(values):
    head=None
    for v in reversed(values):
        head=TreeNode(v,None,head)
    return head

for n in range(0,20):
    arr=list(range(n))
    for tree in [make_left_chain(arr), make_right_chain(arr)]:
        assert inorder_recursive(tree)==inorder_iterative(tree)
print("PASS 40 ca đối chiếu recursive ↔ iterative")

## 16. Độ phức tạp — cần phân biệt output và extra space

Gọi `n` là số node, `h` là **chiều cao của cây** (số node trên một đường đi dài nhất).

- **Time:** O(n) với cả hai cách, vì mỗi node được thăm đúng một lần (iterative: push một lần, pop một lần).
- **Extra Space (không tính mảng kết quả):** O(h) cho **độ sâu đệ quy** hoặc **stack**. Cây cân bằng có `h=O(log n)`, cây lệch có `h=O(n)`.
- **Output Space:** list kết quả chứa n giá trị, cần **O(n)**.

Đệ quy có thể vượt giới hạn call stack của Python trên cây quá sâu; iterative stack tránh giới hạn đệ quy đó.

## 17. Sai lầm dễ gặp

1. `append(node.val)` **trước** khi duyệt trái → đó là preorder, không phải inorder.
2. `while current is not None:` mà quên `or stack` → dừng ngay khi đã đi tới nhánh trái rỗng.
3. Pop xong quên chuyển `current=current.right` → mất toàn bộ nhánh phải.
4. `stack.pop(0)` thay vì `stack.pop()` → lấy đầu list, không còn LIFO và tốn thời gian dịch các phần tử.
5. Nghĩ inorder luôn tăng dần → chỉ có đảm bảo này với **BST hợp lệ**, không phải mọi binary tree.
6. Không xử lý `root=None` → AttributeError.

In [ ]:
not_bst=TreeNode(1,TreeNode(9),TreeNode(0))
print("Inorder cây bất kỳ:",inorder_iterative(not_bst))
print("Kết quả KHÔNG nhất thiết tăng dần")
print("Cây rỗng:",inorder_iterative(None))

## 18. Nhận diện pattern và biến thể

Nếu đề nói **binary tree, traversal, DFS**, hỏi thứ tự thăm node:
- Preorder: **Node → Left → Right**.
- Inorder: **Left → Node → Right**.
- Postorder: **Left → Right → Node**.

Với BST, inorder giúp lấy các giá trị theo thứ tự đã sắp. Với cây bất kỳ, nó chỉ phản ánh cấu trúc trái/phải.

## 19. Năm bài tập tự luyện

Tự làm trước, gợi ý theo ba cấp. Đáp án ở cuối, không gọi hàm TODO.

### Bài 1 — Đọc con trái · Tự làm

Với root có left, trả root.left.val.

**Gợi ý cấp 1 — tư duy:** Con trái là một node.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Truy cập `.left.val`.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

return root.left.val.

</details>

In [ ]:
def exercise_94_1(root: TreeNode)->int:
    # TODO
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_94_1(TreeNode(2,TreeNode(1)))==1

### Bài 2 — Thao tác stack · Tự làm

Cho list `stack=[1,2,3]`, trả phần tử pop từ cuối.

**Gợi ý cấp 1 — tư duy:** Stack LIFO.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng `.pop()` (không có tham số).

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

return stack.pop().

</details>

In [ ]:
def exercise_94_2(stack:list[int])->int:
    # TODO
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_94_2([1,2,3])==3

### Bài 3 — Tính tay · Tự làm

Cây root=2, left=1, right=3. Trả list inorder.

**Gợi ý cấp 1 — tư duy:** Trái trước gốc, phải sau.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Đọc lần lượt node trái/cha/phải.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

[1,2,3]

</details>

In [ ]:
def exercise_94_3()->list[int]:
    # TODO: tự tính
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_94_3()==[1,2,3]

### Bài 4 — Tự viết inorder đệ quy · Tự làm

Không gọi inorder_recursive; tự viết DFS trái→node→phải.

**Gợi ý cấp 1 — tư duy:** Node None là điểm dừng.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng result=[] rồi hàm visit(node) lồng nhau.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

visit(node.left); result.append(node.val); visit(node.right).

</details>

In [ ]:
def exercise_94_4(root: Optional[TreeNode])->list[int]:
    # TODO: đệ quy
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_94_4(TreeNode(2,TreeNode(1),TreeNode(3)))==[1,2,3]

### Bài 5 — Tự viết inorder iterative · Tự làm

Không gọi inorder_iterative; dùng list làm stack.

**Gợi ý cấp 1 — tư duy:** Đẩy hết các node trên nhánh trái.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

while current is not None or stack.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Nếu current: push và đi trái; nếu không: pop, ghi, đi phải.

</details>

In [ ]:
def exercise_94_5(root: Optional[TreeNode])->list[int]:
    # TODO: stack
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_94_5(TreeNode(1,None,TreeNode(2,TreeNode(3))))==[1,3,2]

## 20. Đáp án tham khảo — chỉ đọc sau khi tự làm

In [ ]:
def answer_94_1(root): return root.left.val

def answer_94_2(stack): return stack.pop()

def answer_94_3(): return [1,2,3]

def answer_94_4(root):
    out=[]
    def dfs(node):
        if node is None:return
        dfs(node.left)
        out.append(node.val)
        dfs(node.right)
    dfs(root)
    return out

def answer_94_5(root):
    out=[];stack=[];current=root
    while current is not None or stack:
        if current is not None:
            stack.append(current)
            current=current.left
        else:
            current=stack.pop()
            out.append(current.val)
            current=current.right
    return out

assert answer_94_1(TreeNode(2,TreeNode(1)))==1
assert answer_94_2([1,2,3])==3
assert answer_94_3()==[1,2,3]
assert answer_94_4(TreeNode(2,TreeNode(1),TreeNode(3)))==[1,2,3]
assert answer_94_5(TreeNode(1,None,TreeNode(2,TreeNode(3))))==[1,3,2]
print("PASS tất cả đáp án bài tập #94")

## Tóm tắt — mẫu tư duy cần nhớ

**Inorder = LEFT → NODE → RIGHT.** Đệ quy gọi left, ghi node, gọi right. Với stack: **push chuỗi node bên trái**, khi hết thì **pop, ghi, đi phải**. **Time O(n); extra space O(h) ngoài kết quả O(n).**

## Checklist trước khi submit

- [ ] Tôi giải thích được `root`, `left`, `right`, `None`.
- [ ] Tôi phân biệt inorder và preorder.
- [ ] Tôi trace được push/pop và lúc ghi node.
- [ ] Tôi xử lý `root=None` và cây lệch.
- [ ] Tôi phân tích Time O(n), extra Space O(h) (+ output O(n)).
- [ ] Tôi nộp `Solution.inorderTraversal`.